In [19]:
#import os
import pandas as pd
import numpy as np
import re

# import pypandoc


import sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk

# import docx
# from docx import Document
import os

from sklearn.feature_extraction.text import CountVectorizer
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment import SentimentIntensityAnalyzer

from nltk.stem.wordnet import WordNetLemmatizer

# Initialize the lemmatizer
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/kla21002/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [20]:
# # "C:\Users\Claudia\University of Connecticut\strategic_plans - Documents\data\clean\documents_df.csv"
PATH = "/Users/kla21002/Library/CloudStorage/OneDrive-UniversityofConnecticut/Documents - strategic_plans/"
DATA_PATH = PATH + "data/"
RESULTS_PATH = PATH + "results/"

In [21]:
directory_path = os.getcwd()  # Get the current directory path
directory_path

'/Users/kla21002/Library/CloudStorage/OneDrive-UniversityofConnecticut/Documents - strategic_plans/code'

In [22]:
#for a list of string .lower() if for a list of strings not tokens

def clean_text(
    text, 
    remove_stopwords=False, 
    languages=('english',),  # Tuple of languages to handle
    combine_month_year=True,
    custom_stopwords=None, 
    remove_numbers=True, 
    remove_unusual=True, 
    remove_single_letters=True,
    lemmatize=False, 
    remove_months=False,
    remove_urls = True,
    remove_repeated_characters=False,
    apply_min_df=False,  # Whether to filter tokens by document frequency
    min_df=1,  # Minimum document frequency as a proportion or absolute count
    token_doc_counts=None  # A dictionary to store token document frequencies

):
    """
    Clean and tokenize text into words, with optional removal of stopwords (English, Spanish, etc.),
    numeric tokens, unusual words, and single-letter tokens.
    
    Args:
        text (str): The input text to be cleaned and tokenized.
        remove_stopwords (bool): Whether to remove stopwords. Default is True.
        languages (tuple): Languages for stopwords. Default is ('english',).
        custom_stopwords (set): Optional set of custom stopwords to remove.
        remove_numbers (bool): Whether to remove numeric tokens. Default is True.
        remove_unusual (bool): Whether to remove unusual words. Default is True.
        remove_single_letters (bool): Whether to remove single-letter tokens. Default is True.
    
    Returns:
        list: A list of cleaned and tokenized words.
    """
    # Step 1: Lowercase the text
    text = text.lower()

    # Step 2: Remove punctuation and special characters
    text = re.sub(r'[^\w\s]', '', text)

    if combine_month_year:
        # Regular expression to match a month followed by a year
        month_year_pattern = r'\b(january|february|march|april|may|june|july|august|september|october|november|december)\s(\d{4})\b'
        # Replace matched patterns with the month and year combined with an underscore
        text = re.sub(month_year_pattern, r'\1_\2', text, flags=re.IGNORECASE)
    
    # Step 3: Tokenize the text
    tokens = word_tokenize(text)

    # Step 4: Remove stopwords (if enabled)
    if remove_stopwords:
        stop_words = set()
        for lang in languages:
            stop_words = stop_words.union(set(stopwords.words(lang)))
        if custom_stopwords:
            stop_words = stop_words.union(custom_stopwords)
        tokens = [word for word in tokens if word not in stop_words]
    
    # Step 5: Remove numeric tokens (if enabled)
    if remove_numbers:
        tokens = [word for word in tokens if not word.isdigit()]
    
    # Step 6: Remove unusual words (if enabled)
    if remove_unusual:
        tokens = [word for word in tokens if re.match("^[A-Za-z]+$", word)]
    
    # Step 7: Remove single-letter tokens (if enabled)
    if remove_single_letters:
        tokens = [token for token in tokens if len(token) > 2]
    
    if remove_months:
        months = {
            "january", "february", "march", "april", "may", "june",
            "july", "august", "september", "october", "november", "december",
            "jan", "feb", "mar", "apr", "jun", "jul", "aug", "sep", "oct", "nov", "dec"
        }
        tokens = [token for token in tokens if token not in months]


    if remove_urls:
        tokens = [token for token in tokens if not (token.startswith("http") or token.startswith("www"))]
     # Step 8: Lemmatize tokens (if enabled)
    # if lemmatize:
    #     tokens = lemmatize_tokens(tokens)

    # if remove_repeated_characters:
    #     text = re.sub(r'(.)\1{2,}', r'\1', text)

    #  # Step 10: Apply min_df filtering (if enabled)
    # if apply_min_df and token_doc_counts is not None:
    #     total_docs = token_doc_counts.get('_total_docs', 1)  # Get total docs if proportion used
    #     tokens = [
    #         token for token in tokens
    #         if (token_doc_counts.get(token, 0) >= (min_df if isinstance(min_df, int) else min_df * total_docs))
    #     ]
    
    string = ' '.join(tokens)
    return string


# Import

In [23]:
#remove failed_parse = 1
#we want 0
documents_df = pd.read_csv('documents_df.csv', delimiter='|')
documents_df.head()

,district,pages,ocr,text,filename,contains_alphanumeric,failed_parse
0,Smethport Area SD,9,0,Smethport Area SD District Level Plan 07/01/20...,Smethport Area SD.csv,True,0
1,Bibb County,22,0,#Built4Bibb: More Victory Planned 2023-2028 St...,Bibb County.csv,True,0
2,Fairview SD 72,1,0,Community Connections and Relations - Develop ...,Fairview SD 72.csv,True,0
3,SLATON ISD,40,0,Slaton Independent School District District Im...,SLATON ISD.csv,True,0
4,Springfield,45,0,Reimagining School to Realize the Portrait of ...,Springfield.csv,True,0


# Basic Cleaning

In [24]:

# Create a new column 'text_lower' with lowercase and no punctuation
documents_df['text_lower'] = documents_df['text'].apply(
    lambda x: re.sub(r'[^\w\s]', '', x.lower())  # Lowercase and remove punctuation
)

# Display the updated DataFrame
documents_df.head()

,district,pages,ocr,text,filename,contains_alphanumeric,failed_parse,text_lower
0,Smethport Area SD,9,0,Smethport Area SD District Level Plan 07/01/20...,Smethport Area SD.csv,True,0,smethport area sd district level plan 07012020...
1,Bibb County,22,0,#Built4Bibb: More Victory Planned 2023-2028 St...,Bibb County.csv,True,0,built4bibb more victory planned 20232028 strat...
2,Fairview SD 72,1,0,Community Connections and Relations - Develop ...,Fairview SD 72.csv,True,0,community connections and relations develop c...
3,SLATON ISD,40,0,Slaton Independent School District District Im...,SLATON ISD.csv,True,0,slaton independent school district district im...
4,Springfield,45,0,Reimagining School to Realize the Portrait of ...,Springfield.csv,True,0,reimagining school to realize the portrait of ...


In [25]:
documents_df['text_clean'] = documents_df['text_lower'].apply(clean_text)
documents_df.head(10)

LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - '/Users/kla21002/nltk_data'
    - '/Users/kla21002/opt/anaconda3/envs/strategic_plans/nltk_data'
    - '/Users/kla21002/opt/anaconda3/envs/strategic_plans/share/nltk_data'
    - '/Users/kla21002/opt/anaconda3/envs/strategic_plans/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


In [ ]:
documents_df[["text_clean"]].sample(5)

,text_clean
188,hardin northern local schools strategic plan s...
13,north ammon road idaho falls idaho fax scott w...
499,strategic planning our district home strategic...
94,grading learning committee committee members b...
373,galton ara ridge treet comprehensive plan miio...


# Optional Cleaning

## DIY N-Grams

In [ ]:
j_bgrams = pd.read_excel('diy_ngram_list_clean.xlsx')

In [ ]:
j_bgrams.head()

,diy_ngrams
0,504
1,21st century skills
2,21st century skill
3,5 c
4,5 c s


In [ ]:
# Convert the relevant column (e.g., 'diy_ngrams') to a Python list
diy_bigrams = j_bgrams['diy_ngrams'].dropna().tolist()  # Drop any empty row
diy_bigrams 

[504,
 '21st century skills ',
 '21st century skill',
 '5 c ',
 '5 c s',
 '504 plan',
 '504 plans',
 'academic achievement',
 'academic assistance',
 'academic growth',
 'academic proficiency ',
 'academic remediation',
 'achievement gap',
 'achievement gaps',
 'act',
 'active learners',
 'active lifestyle',
 'advanced placement',
 'after school ',
 'alcohol free',
 'aligned curriculum',
 'alternative diploma',
 'alternative school ',
 'alternative schooling',
 'american indian',
 'ap',
 'assistant teacher',
 'assistant teachers',
 'at risk ',
 'attendance rate',
 'attendance rates',
 'behavior interventions and supports',
 'behavior support',
 'behavioral challenges',
 'behavioral issues',
 'behavioral referral',
 'behavioral referrals',
 'behavioral well being',
 'birth to 3',
 'birth to pre k ',
 'bullying prevention ',
 'bus driver',
 'bus drivers',
 'business partnerships',
 'career education ',
 'career fair',
 'career pathways',
 'career plan',
 'career planning',
 'career prep 

In [ ]:
#diy_bigrams = [bigram.strip().replace(" ", "_") for bigram in diy_bigrams]
#diy_bigrams
# Replace spaces with underscores, filtering non-string values
diy_bigrams = [str(bigram).strip().replace(" ", "_") for bigram in diy_bigrams if isinstance(bigram, str)]

# Display the updated list
print(diy_bigrams)

['21st_century_skills', '21st_century_skill', '5_c', '5_c_s', '504_plan', '504_plans', 'academic_achievement', 'academic_assistance', 'academic_growth', 'academic_proficiency', 'academic_remediation', 'achievement_gap', 'achievement_gaps', 'act', 'active_learners', 'active_lifestyle', 'advanced_placement', 'after_school', 'alcohol_free', 'aligned_curriculum', 'alternative_diploma', 'alternative_school', 'alternative_schooling', 'american_indian', 'ap', 'assistant_teacher', 'assistant_teachers', 'at_risk', 'attendance_rate', 'attendance_rates', 'behavior_interventions_and_supports', 'behavior_support', 'behavioral_challenges', 'behavioral_issues', 'behavioral_referral', 'behavioral_referrals', 'behavioral_well_being', 'birth_to_3', 'birth_to_pre_k', 'bullying_prevention', 'bus_driver', 'bus_drivers', 'business_partnerships', 'career_education', 'career_fair', 'career_pathways', 'career_plan', 'career_planning', 'career_prep', 'career_readiness', 'career_ready', 'career_ready_skills', 'c

In [ ]:
diy_bigrams

['21st_century_skills',
 '21st_century_skill',
 '5_c',
 '5_c_s',
 '504_plan',
 '504_plans',
 'academic_achievement',
 'academic_assistance',
 'academic_growth',
 'academic_proficiency',
 'academic_remediation',
 'achievement_gap',
 'achievement_gaps',
 'act',
 'active_learners',
 'active_lifestyle',
 'advanced_placement',
 'after_school',
 'alcohol_free',
 'aligned_curriculum',
 'alternative_diploma',
 'alternative_school',
 'alternative_schooling',
 'american_indian',
 'ap',
 'assistant_teacher',
 'assistant_teachers',
 'at_risk',
 'attendance_rate',
 'attendance_rates',
 'behavior_interventions_and_supports',
 'behavior_support',
 'behavioral_challenges',
 'behavioral_issues',
 'behavioral_referral',
 'behavioral_referrals',
 'behavioral_well_being',
 'birth_to_3',
 'birth_to_pre_k',
 'bullying_prevention',
 'bus_driver',
 'bus_drivers',
 'business_partnerships',
 'career_education',
 'career_fair',
 'career_pathways',
 'career_plan',
 'career_planning',
 'career_prep',
 'career_read

In [ ]:
# Prepare old_terms and new_terms from diy_bigrams
old_terms = [bigram.replace("_", " ") for bigram in diy_bigrams]  # Convert underscores to spaces
new_terms = diy_bigrams  # Keep the underscore-separated format

In [ ]:
# Create a new column 'text_with_bigrams' to store the updated text
documents_df['clean_text_diy_grams'] = documents_df['text_clean']

# Replace old terms with new terms using a loop
for old_term, new_term in zip(old_terms, new_terms):
    documents_df['clean_text_diy_grams'] = documents_df['clean_text_diy_grams'].str.replace(
        old_term, new_term, regex=False  # Use regex=False to treat terms literally
    )
documents_df.sample()

,district,pages,ocr,text,filename,contains_alphanumeric,failed_parse,text_lower,text_clean,clean_text_diy_grams
439,San Jose Unified,2,0,Strategic Plan San Jos Unieds strategic plan w...,San Jose Unified.csv,True,0,strategic plan san jos unieds strategic plan w...,strategic plan san jos unieds strategic plan w...,strategic_plan san jos unieds strategic_plan w...


In [ ]:
# import re

# def replace_with_bigrams(text, bigrams):
#     """
#     Replace words/phrases in the text with their bigram forms.
    
#     Args:
#         text (str): Input text to process.
#         bigrams (list): List of bigram phrases to identify in the text.
    
#     Returns:
#         str: Text with bigram phrases replaced.
#     """
#     # Sort bigrams by length (longest first) to prevent partial replacements
#     bigrams = sorted(bigrams, key=len, reverse=True)
#     for bigram in bigrams:
#         # Replace spaces with underscores in the text to match bigram format
#         pattern = re.escape(bigram.replace("_", " "))
#         text = re.sub(rf'\b{pattern}\b', bigram, text)
#     return text


In [ ]:
documents_df['clean_text_diy_grams'][6]

'newcastle school district improvement plan enter learn forth serve adopted newcastle isd board trustees updated the policy newcastle isd not discriminate the basis race color national origin gender handicap its vocational programs services activities required title the civil rights act amended title the education amendments and sections and the rehabilitation act amended newcastle isd will take steps ensure that lack english language skills will not barrier admission and participation all educational programs and services mission statement newcastle isd students will become lifelong learners who develop personal knowledge skills and competence maximum capacity and learn behavior patterns which will make each student responsible member society terms their individual abilities all students should achieve long range goals raise the level achievement for all students ensure greater number economically_disadvantaged students further their education foster schoolwide love for reading and in

In [ ]:
#         text (str): Input text.
    
#     Returns:
#         str: Updated text with month-year combinations.
#     """
#     # Regular expression to match a month followed by a year
#     month_year_pattern = r'\b(january|february|march|april|may|june|july|august|september|october|november|december)\s(\d{4})\b'
#     # Replace matched patterns with the month and year combined with an underscore
#     return re.sub(month_year_pattern, r'\1_\2', text, flags=re.IGNORECASE)

# # Apply the function to the 'text_with_bigrams' column
# documents_df['text_with_j_grams'] = documents_df['text_with_j_grams'].apply(combine_month_year)
# # documents_df['text_with_j_grams'] = documents_df['text_with_j_grams'].apply(combine_month_year)

# # Display the updated DataFrame
# documents_df.head()

# Tokenize

In [15]:
from nltk.tokenize import word_tokenize

In [16]:
#lets tokenize
documents_df['tokens_no_grams'] = documents_df['text_clean'].apply(word_tokenize)
documents_df['tokens_diy_grams'] = documents_df['clean_text_diy_grams'].apply(word_tokenize)

documents_df.head()

KeyError: 'text_clean'

# Optional cleaning post tokens

In [51]:
len(documents_df)

615

In [ ]:

#texts have been tokenized before doing the lemi
def lemmatize_tokens(token_list):
    """
    Lemmatize a list of tokens using NLTK's WordNetLemmatizer.
    """
    new_tokens = [lemmatizer.lemmatize(token) for token in token_list]
    return new_tokens

In [96]:
def clean_and_tokenize(
    tokens, 
    remove_stopwords=True, 
    languages=('english',),  # Tuple of languages to handle
    custom_stopwords=None, 
    remove_numbers=True, 
    remove_unusual=True, 
    remove_single_letters=True,
    remove_urls= True,
    lemmatize=True, 
    remove_months=True,
    remove_repeated_characters=True,
    apply_min_df=False,  # Whether to filter tokens by document frequency
    min_df=0.20,  # Minimum document frequency as a proportion or absolute count
    token_doc_counts=None  # A dictionary to store token document frequencies
):
    """
    Clean and process a list of tokens.
    
    Args:
        tokens (list): Tokenized input text.
        remove_stopwords (bool): Whether to remove stopwords. Default is True.
        languages (tuple): Languages for stopwords. Default is ('english',).
        custom_stopwords (set): Optional set of custom stopwords to remove.
        remove_numbers (bool): Whether to remove numeric tokens. Default is True.
        remove_unusual (bool): Whether to remove unusual words. Default is True.
        remove_single_letters (bool): Whether to remove single-letter tokens. Default is True.
    
    Returns:
        list: A list of cleaned and processed tokens.
    """
    # Step 1: Remove stopwords (if enabled)
    if remove_stopwords:
        stop_words = set()
        for lang in languages:
            stop_words = stop_words.union(set(stopwords.words(lang)))
        if custom_stopwords:
            stop_words = stop_words.union(custom_stopwords)
        tokens = [word for word in tokens if word not in stop_words]
    
    # Step 2: Remove numeric tokens (if enabled) #remove entirely numeric tokens only
    if remove_numbers:
        tokens = [word for word in tokens if not word.isdigit()]
    
    # Step 3: Remove unusual words (if enabled)
    #if remove_unusual:
        #tokens = [word for word in tokens if re.match("^[A-Za-z]+$", word)] # remove tokens with _, numeric in it we need to update
    if remove_unusual:
        tokens = [word for word in tokens if re.match("^[A-Za-z0-9_]+$", word)]
    
    # Step 4: Remove single-letter tokens (if enabled)
    if remove_single_letters:
        tokens = [token for token in tokens if len(token) > 2]
    
    # Step 5: Remove tokens containing URLs (if enabled)
    if remove_urls:
        tokens = [token for token in tokens if not (token.startswith("http") or token.startswith("www"))]
    
    # Step 6: Remove months (if enabled)
    if remove_months:
        months = {
            "january", "february", "march", "april", "may", "june",
            "july", "august", "september", "october", "november", "december",
            "jan", "feb", "mar", "apr", "jun", "jul", "aug", "sep", "oct", "nov", "dec"
        }
        tokens = [token for token in tokens if token not in months]

    # Step 6: Lemmatize tokens (if enabled)
    if lemmatize:
        tokens = lemmatize_tokens(tokens)

    # Step 7: Remove repeated characters (if enabled)
   # if remove_repeated_characters:
   #     tokens = [re.sub(r'(.)\1{2,}', r'\1', token) for token in tokens]

    # Step 8: Apply min_df filtering (if enabled)
   # if apply_min_df and token_doc_counts is not None:
  #      total_docs = token_doc_counts.get('_total_docs', 1)  # Get total docs if proportion used
  #      tokens = [
   #         token for token in tokens
   #         if (token_doc_counts.get(token, 0) >= (min_df if isinstance(min_df, int) else min_df * total_docs))
   #     ]
    
    return tokens


In [97]:
# Example with customized inputs
documents_df['cleaned_tokens'] = documents_df['tokens'].apply(
    lambda x: clean_and_tokenize(
        x, 
        remove_stopwords=True, 
        languages=('english', 'spanish'),  # Example: Removing both English and Spanish stopwords
        custom_stopwords=False,  # Example: Adding custom stopwords #custom_stopwords={'sd', 'pssa'},
        remove_numbers=True, 
        remove_unusual=True, 
        remove_single_letters=True, 
        remove_urls= True,
        lemmatize=True,  # Enable lemmatization
        remove_months=False, #keep the years or moths keeps them, false
        remove_repeated_characters=True,
    )
)


In [99]:
documents_df
# Create a subset of the DataFrame
subset_df = documents_df[['district', 'text', 'text_lower', 'text_with_bigrams', 'cleaned_tokens']]

subset_df

,district,text,text_lower,text_with_bigrams,cleaned_tokens
0,Smethport Area SD,Smethport Area SD District Level Plan 07/01/20...,smethport area sd district level plan 07012020...,smethport area sd district level plan 07012020...,"[smethport, area, district, level, plan, distr..."
1,Bibb County,#Built4Bibb: More Victory Planned 2023-2028 St...,built4bibb more victory planned 20232028 strat...,built4bibb more victory planned 20232028 strat...,"[built4bibb, victory, planned, strategic, plan..."
2,Fairview SD 72,Community Connections and Relations - Develop ...,community connections and relations develop c...,community connections and relations develop c...,"[community, connection, relation, develop, com..."
3,SLATON ISD,Slaton Independent School District District Im...,slaton independent school district district im...,slaton independent school district district im...,"[slaton, independent, school, district, distri..."
4,Springfield,Reimagining School to Realize the Portrait of ...,reimagining school to realize the portrait of ...,reimagining school to realize the portrait of ...,"[reimagining, school, realize, portrait, gradu..."
...,...,...,...,...,...
610,Germantown,S T R A T E G I C P L A N 2 0 2 5 I N S P I R ...,s t r a t e g i c p l a n 2 0 2 5 i n s p i r ...,s t r a t e g i c p l a n 2 0 2 5 i n s p i r ...,"[board, member, rebecca, luter, linda, fisher,..."
611,Sunnyvale,"Page 83 of 216 Goals, Actions, & Services Stra...",page 83 of 216 goals actions services strateg...,page 83 of 216 goals actions services strateg...,"[page, goal, action, service, strategic, plann..."
612,Big Sky School K-12,"Strategic Area: Flexible Pathways, Dynamic Pro...",strategic area flexible pathways dynamic progr...,strategic area flexible pathways dynamic progr...,"[strategic, area, flexible, pathway, dynamic, ..."
613,KANSAS CITY 33,MOVING FORWARD TOGETHER 2018-2023 Strategic Pl...,moving forward together 20182023 strategic pla...,moving forward together 20182023 strategic pla...,"[moving, forward, together, strategic, plan, c..."


In [90]:
display(len(subset_df['cleaned_tokens'][3]))
display(subset_df['cleaned_tokens'][3])

['slaton',
 'independent',
 'school',
 'district',
 'district',
 'improvement',
 'plan',
 'slaton',
 'independent',
 'school',
 'district',
 'generated',
 'plan4learningcom',
 'district',
 'february',
 'mission',
 'statement',
 'mission',
 'slaton',
 'isd',
 'inspire',
 'empower',
 'student',
 'lead',
 'extraordinary',
 'life',
 'embrace',
 'possibility',
 '21st',
 'century',
 'relevant',
 'engaging',
 'learning',
 'experience',
 'led',
 'inspirational',
 'nurturing',
 'educator',
 'slaton',
 'independent',
 'school',
 'district',
 'generated',
 'plan4learningcom',
 'district',
 'february',
 'goal',
 'goal',
 'may_2023',
 'slaton',
 'isd',
 'develop',
 'culture',
 'interdisciplinary',
 'literacy',
 'ensuring',
 'reading',
 'writing',
 'thinking',
 'problem_solving',
 'skill',
 'become',
 'part',
 'content',
 'area',
 'instruction',
 'evaluated',
 'predominance',
 'student',
 'work',
 'product',
 'showing',
 'evidence',
 'instructional',
 'criterion',
 'aligned',
 'literacy',
 'strategy

In [86]:
subset_df['cleaned_tokens'][2]
row_index = 3  # Replace with your desired row index
underscore_tokens = [token for token in subset_df['cleaned_tokens'][row_index] if "_" in token]
underscore_tokens

['may_2023',
 'problem_solving',
 'achievement_gap',
 'special_education',
 'economically_disadvantaged',
 'english_language_learners',
 'special_education',
 'problem_solving',
 'professional_development',
 'after_school',
 'instructional_strategies',
 'instructional_practices',
 'progress_monitoring',
 'response_to_intervention',
 'may_2023',
 'problem_solving',
 'academic_achievement',
 'special_education',
 'special_education',
 'digital_learning',
 'professional_development',
 'after_school',
 'digital_learning',
 'special_education',
 'instructional_practices',
 'progress_monitoring',
 'response_to_intervention',
 'may_2023',
 'problem_solving',
 'professional_development',
 'after_school',
 'digital_learning',
 'may_2023',
 'problem_solving',
 'instructional_strategies',
 'after_school',
 'professional_development',
 'may_2023',
 'problem_solving',
 'career_readiness',
 'dual_credit',
 'professional_development',
 'may_2023',
 'problem_solving',
 '_85',
 'dual_credit',
 'dual_cr

# Look at Parameters

In [7]:
parameters_df = pd.read_csv('hyperparameters_models.csv', delimiter=',')
parameters_df.head()

,Unnamed: 0,chunk,stop,diy_gram,tribigram,min_df,lemma,tfidf,topics
0,0,0,0,0,0,1,0,0,10
1,1,0,0,0,0,1,0,0,80
2,2,0,0,0,0,1,0,1,10
3,3,0,0,0,0,1,0,1,80
4,4,0,0,0,0,1,1,0,10


In [30]:
parameters_df.chunk.value_counts()

chunk
0      192
100    192
150    192
200    192
Name: count, dtype: int64

notes
* remove stop words
* diy_gram julia words
* stop 1 or 0 whatever or not to remove stopwords
* min_df

In [ ]:
parameters_test = parameters_df.sample(50,random_state= 10)
parameters_test

In [9]:
len(documents_df)

615

#Create a code to chunk 100, 150, 200 create a code

##Doc 100


In [10]:
#if 'text_lower' not in documents_df.columns:
    #documents_df['text_lower'] = documents_df['text'].str.lower()

In [11]:
#documents_df['text_lower'] = documents_df['text'].str.lower()

In [100]:
subset_df.head()

,district,text,text_lower,text_with_bigrams,cleaned_tokens
0,Smethport Area SD,Smethport Area SD District Level Plan 07/01/20...,smethport area sd district level plan 07012020...,smethport area sd district level plan 07012020...,"[smethport, area, district, level, plan, distr..."
1,Bibb County,#Built4Bibb: More Victory Planned 2023-2028 St...,built4bibb more victory planned 20232028 strat...,built4bibb more victory planned 20232028 strat...,"[built4bibb, victory, planned, strategic, plan..."
2,Fairview SD 72,Community Connections and Relations - Develop ...,community connections and relations develop c...,community connections and relations develop c...,"[community, connection, relation, develop, com..."
3,SLATON ISD,Slaton Independent School District District Im...,slaton independent school district district im...,slaton independent school district district im...,"[slaton, independent, school, district, distri..."
4,Springfield,Reimagining School to Realize the Portrait of ...,reimagining school to realize the portrait of ...,reimagining school to realize the portrait of ...,"[reimagining, school, realize, portrait, gradu..."


In [101]:
def split_tokens_into_chunks(tokens, chunk_size=100):
    """
    Split a list of tokens into chunks of a fixed size.
        tokens (list): List of tokens to split.
        chunk_size (int): The number of tokens per chunk.
    
    Returns:
        List of token chunks as strings.
    """
    # Create chunks of the specified size
    chunks_split = [tokens[i:i + chunk_size] for i in range(0, len(tokens), chunk_size)]
    return [' '.join(chunk_split) for chunk_split in chunks_split]


In [102]:
# Create new documents with tokenized chunks
new_documents = []

for index, row in documents_df.iterrows():
    # Use the cleaned_tokens column to split into chunks
    cleaned_tokens = row['cleaned_tokens']
    
    # Split the tokens into chunks
    chunks = split_tokens_into_chunks(cleaned_tokens, chunk_size=100)
    
    # Add each chunk as a new document
    for i, chunk in enumerate(chunks):
        new_document = {
            'district': row['district'],
            'chunk_index': i + 1,
            'chunk_text': chunk  # Join tokens into a single string for each chunk
        }
        new_documents.append(new_document)

# Create a new DataFrame with the split documents
chunked_documents_df = pd.DataFrame(new_documents)

# Display the resulting DataFrame
print(chunked_documents_df.head())


            district  chunk_index  \
0  Smethport Area SD            1   
1  Smethport Area SD            2   
2  Smethport Area SD            3   
3  Smethport Area SD            4   
4  Smethport Area SD            5   

                                          chunk_text  
0  smethport area district level plan district le...  
1  math ela science pssa keystone advanced rate s...  
2  accomplishment plan strategy common assessment...  
3  validation differentiated instruction package ...  
4  school implementation step teacher team evalua...  


In [108]:
#len(chunked_documents_df['chunk_text'][0]) this calculate the number of characters not words (e.g. house, 5 characters)
len(chunked_documents_df['chunk_text'][0].split())

100

The whole function

In [113]:
def create_chunked_dataframe_from_tokens(documents_df, chunk_size):
    """
    Create a DataFrame with tokenized text split into chunks of a fixed size.
    
    Args:
        documents_df (pd.DataFrame): The input DataFrame with a 'cleaned_tokens' column.
        chunk_size (int): The desired number of tokens per chunk.
    
    Returns:
        pd.DataFrame: A DataFrame with chunked text and metadata.
    """
    def split_tokens_into_fixed_chunks(tokens, chunk_size):
        """
        Split a list of tokens into chunks of a fixed size.
        """
        # Create chunks with the specified chunk size
        chunks_split = [tokens[i:i + chunk_size] for i in range(0, len(tokens), chunk_size)]
        # Join tokens in each chunk into a string
        return [' '.join(chunk_split) for chunk_split in chunks_split]
    
    # Initialize an empty list to store new documents
    new_documents = []

    # Loop through the DataFrame to split tokens into chunks
    for index, row in documents_df.iterrows():
        cleaned_tokens = row['cleaned_tokens']
        # Split the tokens into chunks of the specified size
        chunks = split_tokens_into_fixed_chunks(cleaned_tokens, chunk_size)
        
        # Add each chunk to the new documents list with metadata
        for i, chunk in enumerate(chunks):
            new_document = {
                'district': row['district'],
                'chunk_index': i + 1,
                'chunk_text': chunk
            }
            new_documents.append(new_document)

    # Create a new DataFrame from the new documents list
    new_df = pd.DataFrame(new_documents)
    return new_df


In [114]:
# Example usage
chunk_size = 50
new_df_100 = create_chunked_dataframe_from_tokens(subset_df, chunk_size)
print(new_df_100.head())

            district  chunk_index  \
0  Smethport Area SD            1   
1  Smethport Area SD            2   
2  Smethport Area SD            3   
3  Smethport Area SD            4   
4  Smethport Area SD            5   

                                          chunk_text  
0  smethport area district level plan district le...  
1  math ela science pssa keystone advanced rate s...  
2  accomplishment plan strategy common assessment...  
3  validation differentiated instruction package ...  
4  school implementation step teacher team evalua...  


In [ ]:
len(new_df_100['chunk_text'][0].split())

In [49]:
#CHUNK codefirst chunk will be 100, this need string list!!!!!!!!!!!!!!
#second chunk 200
def split_text_into_chunks(text):
    words = text.split()
    chunks_split = [words[i:i + 100] for i in range(0, len(words), 100)]
    return [' '.join(chunk_split) for chunk_split in chunks_split]

# Create new documents
new_documents = []

for index, row in documents_df.iterrows():
    text_lower = row['text_lower']    # text_lower = row['text_lower'] 
    chunks = split_text_into_chunks(text_lower)
    
    for i, chunk in enumerate(chunks):
        new_document = {
            'district': row['district'],
            'chunk_index': i + 1,
            'chunk_text': chunk
        }
        new_documents.append(new_document)

In [109]:
chunked_documents_df['chunk_tokens'] = chunked_documents_df['chunk_text'].apply(word_tokenize)

In [110]:
chunked_documents_df['chunk_tokens']

0        [smethport, area, district, level, plan, distr...
1        [math, ela, science, pssa, keystone, advanced,...
2        [accomplishment, plan, strategy, common, asses...
3        [validation, differentiated, instruction, pack...
4        [school, implementation, step, teacher, team, ...
                               ...                        
11494    [kindergarten, outreach, elementary, school, m...
11495    [accelerated, ela, class, grade, metric, devel...
11496    [educator, however, involvement, variety, grou...
11497    [focused, providing, rigorous, relevant, instr...
11498    [valley, regional, career, academy, tabitha, b...
Name: chunk_tokens, Length: 11499, dtype: object

##Clean this

In [26]:
new_documents[1:5]

[{'district': 'Smethport Area SD',
  'chunk_index': 2,
  'chunk_text': 'lesson plans, and content resources) aligned with state standards and fully accessible to teachers and students. indicators of effectiveness: type: annual data source: future ready index, pvaas specific targets: pssa and keystone proficient and advanced rates at or above the pa state average in math, ela and science. pssa and keystone advanced rates at or above the pa state average in math, ela and science. type: annual data source: future ready index, pvaas specific targets: pvaas growth data showing evidence that the district has met the standard for growth in each year (green) and for the 3 year average in'},
 {'district': 'Smethport Area SD',
  'chunk_index': 3,
  'chunk_text': 'pssa grade 4 math, pssa grade 5 math, pssa grade 6 math and pssa grade 7 math. 110 pvaas growth data showing evidence that the district has met the standard for growth in each year (green) and for the 3 year average in pssa grade 5 ela,

In [ ]:
#

In [51]:
new_df = pd.DataFrame(new_documents)
new_df 

,district,chunk_index,chunk_text
0,Smethport Area SD,1,smethport area sd district level plan 07/01/20...
1,Smethport Area SD,2,"lesson plans, and content resources) aligned w..."
2,Smethport Area SD,3,"pssa grade 4 math, pssa grade 5 math, pssa gra..."
3,Smethport Area SD,4,strategies: common assessment within grade/sub...
4,Smethport Area SD,5,student achievement data to support instructio...
...,...,...,...
18850,Davidson County,15,school students metric 3: increase advanced-le...
18851,Davidson County,16,into instructional design metric 1: visual rep...
18852,Davidson County,17,schools. members of our community attended mee...
18853,Davidson County,18,our students. this draft was reviewed and revi...


In [52]:
#strategic plans 
chunk_counts = new_df.groupby('district')['chunk_index'].count().reset_index()

# Rename the count column to 'total_chunks' for clarity
chunk_counts = chunk_counts.rename(columns={'chunk_index': 'total_chunks'})

# Display the result
print(chunk_counts)

                       district  total_chunks
0                   ABILENE ISD             6
1    ALBUQUERQUE PUBLIC SCHOOLS            44
2                      ANADARKO            82
3                  ANGLETON ISD            10
4                ARAPAHO-BUTLER             6
..                          ...           ...
567                   Worcester            41
568         YORK PUBLIC SCHOOLS            56
569                     York 01             9
570                York City SD             8
571             Youngstown City           106

[572 rows x 2 columns]


In [53]:
new_df.value_counts()
len(new_df)

18855

CHUNK SIZE Function

In [ ]:
def split_text_into_fixed_chunks(text, chunk_size):
    """
    Split text into chunks of a fixed size.
    Each chunk will contain exactly 'chunk_size' words, except the last chunk if the total
    word count is not a multiple of chunk_size.
    """
    words = text.split()
    # Create chunks with the specified chunk size
    chunks_split = [words[i:i + chunk_size] for i in range(0, len(words), chunk_size)]
    return [' '.join(chunk_split) for chunk_split in chunks_split]

# Initialize an empty list to store new documents
new_documents = []

# Define the desired chunk size (e.g., 50 words per chunk)
chunk_size = 100

# Loop through the DataFrame to split text into chunks
for index, row in documents_df.iterrows():
    text_lower = row['text_lower']
    # Split the text into chunks of the specified size
    chunks = split_text_into_fixed_chunks(text_lower, chunk_size)
    
    # Add each chunk to the new documents list with metadata
    for i, chunk in enumerate(chunks):
        new_document = {
            'district': row['district'],
            'chunk_index': i + 1,
            'chunk_text': chunk
        }
        new_documents.append(new_document)

# Create a new DataFrame from the new documents list
new_df = pd.DataFrame(new_documents)

# Display the resulting DataFrame
new_df


In [30]:
def split_text_into_custom_chunks(text, num_chunks):
    # Split the text into words
    words = text.split()
    total_words = len(words)
    
    # Calculate the size of each chunk
    chunk_size = max(1, total_words // num_chunks)  # Ensure at least one word per chunk
    
    # Create the chunks
    chunks_split = [words[i:i + chunk_size] for i in range(0, total_words, chunk_size)]
    
    # Join each chunk back into a single string
    return [' '.join(chunk_split) for chunk_split in chunks_split]


In [ ]:
new_documents = []

# Desired number of chunks (can be set dynamically)
desired_chunks = 100

for index, row in documents_df.iterrows():
    text_lower = row['text_lower']
    
    # Use the updated function to split the text into the desired number of chunks
    chunks = split_text_into_custom_chunks(text_lower, desired_chunks)
    
    for i, chunk in enumerate(chunks):
        new_document = {
            'district': row['district'],
            'chunk_index': i + 1,
            'chunk_text': chunk
        }
        new_documents.append(new_document)

# Create a new DataFrame from the chunks
new_df = pd.DataFrame(new_documents)
new_df


In [69]:
new_df.value_counts()
len(new_df)

18855

In [ ]:
def create_chunked_dataframe(documents_df, chunk_size):
    """
    Create a DataFrame with text split into chunks of a fixed size.
    
    Args:
        documents_df (pd.DataFrame): The input DataFrame with a 'text_lower' column.
        chunk_size (int): The desired number of words per chunk.
    
    Returns:
        pd.DataFrame: A DataFrame with chunked text and metadata.
    """
    def split_text_into_fixed_chunks(text, chunk_size):
        """
        Split text into chunks of a fixed size.
        """
        words = text.split()
        # Create chunks with the specified chunk size
        chunks_split = [words[i:i + chunk_size] for i in range(0, len(words), chunk_size)]
        return [' '.join(chunk_split) for chunk_split in chunks_split]
    
    # Initialize an empty list to store new documents
    new_documents = []

    # Loop through the DataFrame to split text into chunks
    for index, row in documents_df.iterrows():
        text_lower = row['text_lower']
        # Split the text into chunks of the specified size
        chunks = split_text_into_fixed_chunks(text_lower, chunk_size)
        
        # Add each chunk to the new documents list with metadata
        for i, chunk in enumerate(chunks):
            new_document = {
                'district': row['district'],
                'chunk_index': i + 1,
                'chunk_text': chunk
            }
            new_documents.append(new_document)

    # Create a new DataFrame from the new documents list
    new_df = pd.DataFrame(new_documents)
    return new_df


In [72]:
def create_chunked_dataframe(documents_df, chunk_size):
    """
    Create a DataFrame with text split into chunks of a fixed size.
    
    Args:
        documents_df (pd.DataFrame): The input DataFrame with a 'text_lower' column.
        chunk_size (int): The desired number of words per chunk.
    
    Returns:
        pd.DataFrame: A DataFrame with chunked text and metadata.
    """
    def split_text_into_fixed_chunks(text, chunk_size):
        """
        Split text into chunks of a fixed size.
        """
        words = text.split()
        # Create chunks with the specified chunk size
        chunks_split = [words[i:i + chunk_size] for i in range(0, len(words), chunk_size)]
        return [' '.join(chunk_split) for chunk_split in chunks_split]
    
    # Initialize an empty list to store new documents
    new_documents = []

    # Loop through the DataFrame to split text into chunks
    for index, row in documents_df.iterrows():
        text_lower = row['text_lower']
        # Split the text into chunks of the specified size
        chunks = split_text_into_fixed_chunks(text_lower, chunk_size)
        
        # Add each chunk to the new documents list with metadata
        for i, chunk in enumerate(chunks):
            new_document = {
                'district': row['district'],
                'chunk_index': i + 1,
                'chunk_text': chunk
            }
            new_documents.append(new_document)

    # Create a new DataFrame from the new documents list
    new_df = pd.DataFrame(new_documents)
    return new_df




In [73]:
# Example usage
chunk_size = 100
new_df_100 = create_chunked_dataframe(documents_df, chunk_size)
print(new_df_100.head())

            district  chunk_index  \
0  Smethport Area SD            1   
1  Smethport Area SD            2   
2  Smethport Area SD            3   
3  Smethport Area SD            4   
4  Smethport Area SD            5   

                                          chunk_text  
0  smethport area sd district level plan 07/01/20...  
1  lesson plans, and content resources) aligned w...  
2  pssa grade 4 math, pssa grade 5 math, pssa gra...  
3  strategies: common assessment within grade/sub...  
4  student achievement data to support instructio...  


In [ ]:
new_df_100

In [77]:
len(new_df_100.iloc[15000]['chunk_text'].split())

100

In [70]:
#len(new_df.iloc[0]['chunk_text'])
len(new_df.iloc[10000]['chunk_text'].split())

100

In [78]:
new_df_100

,district,chunk_index,chunk_text
0,Smethport Area SD,1,smethport area sd district level plan 07/01/20...
1,Smethport Area SD,2,"lesson plans, and content resources) aligned w..."
2,Smethport Area SD,3,"pssa grade 4 math, pssa grade 5 math, pssa gra..."
3,Smethport Area SD,4,strategies: common assessment within grade/sub...
4,Smethport Area SD,5,student achievement data to support instructio...
...,...,...,...
18850,Davidson County,15,school students metric 3: increase advanced-le...
18851,Davidson County,16,into instructional design metric 1: visual rep...
18852,Davidson County,17,schools. members of our community attended mee...
18853,Davidson County,18,our students. this draft was reviewed and revi...


In [95]:
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk

# Ensure NLTK resources are downloaded
nltk.download('punkt')
nltk.download('stopwords')

def clean_and_tokenize(text, remove_stopwords=True, 
                       custom_stopwords=None,
                        remove_numbers=True,
                        remove_unusual=True,
                        remove_single_letters=True):
    """
    Clean and tokenize text into words.
    
    Args:
        text (str): The input text to be cleaned and tokenized.
        remove_stopwords (bool): Whether to remove stopwords. Default is True.
        custom_stopwords (set): Optional set of custom stopwords to remove.
    
    Returns:
        list: A list of cleaned and tokenized words.
    """
    # Step 1: Lowercase the text
    text = text.lower()

    # Step 2: Remove punctuation and special characters
    text = re.sub(r'[^\w\s]', '', text)

    # Step 3: Tokenize the text
    tokens = word_tokenize(text)

    # Step 4: Remove stopwords (if enabled)
    if remove_stopwords:
        stop_words = set(stopwords.words('english'))
        if custom_stopwords:
            stop_words = stop_words.union(custom_stopwords)
        tokens = [word for word in tokens if word not in stop_words]

    # Step 5: Remove numeric tokens (if enabled)
    if remove_numbers:
        tokens = [word for word in tokens if not word.isdigit()]

    # Step 6: Remove unusual words (if enabled)
    if remove_unusual:
        tokens = [word for word in tokens if re.match("^[A-Za-z]+$", word)]

    # Step 7: Remove single-letter tokens (if enabled)
    if remove_single_letters:
        tokens = [token for token in tokens if len(token) > 1]
    
    return tokens




[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Claudia\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Claudia\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [109]:
from nltk.stem.wordnet import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import re
import nltk

# Ensure NLTK resources are downloaded
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

# Define the lemmatizer
lemmatizer = WordNetLemmatizer() #

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Claudia\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Claudia\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Claudia\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


How min_df Works
token_doc_counts:

A dictionary where keys are tokens and values are the number of documents in which the token appears.
Needs to be precomputed across all documents in your corpus.
Absolute vs. Proportional min_df:

If min_df is an integer (e.g., 10), it removes tokens appearing in fewer than 10 documents.
If min_df is a float (e.g., 0.2), it removes tokens appearing in less than 20% of the total documents.
Precomputing Document Frequencies:

Use a preprocessing step to calculate document frequencies for all tokens.


In [156]:
from collections import Counter

In [157]:
# List of tokenized documents
doc_list = list(new_df_100['cleaned_tokens'])

# Count document frequencies
token_doc_counts = Counter()
for doc in doc_list:
    unique_tokens = set(doc)  # Count unique tokens per document
    token_doc_counts.update(unique_tokens)

# Add total document count for proportion-based filtering
token_doc_counts['_total_docs'] = len(doc_list)

In [3]:
# Apply the function with min_df filtering (e.g., 20% of documents)
new_df_100['filtered_tokens'] = new_df_100['cleaned_tokens'].apply(
    lambda x: clean_and_tokenize(
        " ".join(x),  # Rejoin tokens to simulate raw text
        apply_min_df=True,
        min_df=5,  # Token must appear in at least 10% of documents, integer the minimun # of documents
        token_doc_counts=token_doc_counts
    )
)




NameError: name 'new_df_100' is not defined

#i like it with min 0.01

In [174]:
# Display the updated DataFrame
print(new_df_100[['cleaned_tokens', 'filtered_tokens']].head())

                                      cleaned_tokens  \
0  [smethport, area, district, level, plan, distr...   
1  [lesson, plan, content, resource, aligned, sta...   
2  [grade, math, grade, math, grade, math, grade,...   
3  [strategy, common, assessment, within, gradesu...   
4  [student, achievement, data, support, instruct...   

                                     filtered_tokens  
0  [area, district, level, plan, district, level,...  
1  [lesson, plan, content, resource, aligned, sta...  
2  [grade, math, grade, math, grade, math, grade,...  
3  [strategy, common, assessment, within, descrip...  
4  [student, achievement, data, support, instruct...  


In [175]:
new_df_100

,district,chunk_index,chunk_text,cleaned_tokens,filtered_tokens
0,Smethport Area SD,1,smethport area sd district level plan 07/01/20...,"[smethport, area, district, level, plan, distr...","[area, district, level, plan, district, level,..."
1,Smethport Area SD,2,"lesson plans, and content resources) aligned w...","[lesson, plan, content, resource, aligned, sta...","[lesson, plan, content, resource, aligned, sta..."
2,Smethport Area SD,3,"pssa grade 4 math, pssa grade 5 math, pssa gra...","[grade, math, grade, math, grade, math, grade,...","[grade, math, grade, math, grade, math, grade,..."
3,Smethport Area SD,4,strategies: common assessment within grade/sub...,"[strategy, common, assessment, within, gradesu...","[strategy, common, assessment, within, descrip..."
4,Smethport Area SD,5,student achievement data to support instructio...,"[student, achievement, data, support, instruct...","[student, achievement, data, support, instruct..."
...,...,...,...,...,...
18850,Davidson County,15,school students metric 3: increase advanced-le...,"[school, student, metric, increase, advancedle...","[school, student, metric, increase, math, oppo..."
18851,Davidson County,16,into instructional design metric 1: visual rep...,"[instructional, design, metric, visual, repres...","[instructional, design, metric, every, distric..."
18852,Davidson County,17,schools. members of our community attended mee...,"[school, member, community, attended, meeting,...","[school, member, community, meeting, provided,..."
18853,Davidson County,18,our students. this draft was reviewed and revi...,"[student, draft, reviewed, revised, follow, in...","[student, draft, reviewed, revised, group, pro..."


In [123]:
# Clean and tokenize the chunk_text column
new_df_100['cleaned_tokens'] = new_df_100['chunk_text'].apply(clean_and_tokenize)

# Display the updated DataFrame
print(new_df_100[['chunk_text', 'cleaned_tokens']].head())

                                          chunk_text  \
0  smethport area sd district level plan 07/01/20...   
1  lesson plans, and content resources) aligned w...   
2  pssa grade 4 math, pssa grade 5 math, pssa gra...   
3  strategies: common assessment within grade/sub...   
4  student achievement data to support instructio...   

                                      cleaned_tokens  
0  [smethport, area, sd, district, level, plan, d...  
1  [lesson, plan, content, resource, aligned, sta...  
2  [pssa, grade, math, pssa, grade, math, pssa, g...  
3  [strategy, common, assessment, within, gradesu...  
4  [student, achievement, data, support, instruct...  


In [ ]:
new_df_100['cleaned_tokens'][1000]

#bigrams, trigrams

In [1]:
from gensim.models import Phrases

In [3]:
import scipy

In [4]:
from scipy.linalg import triu
print(triu)

ImportError: cannot import name 'triu' from 'scipy.linalg' (C:\Users\Claudia\AppData\Roaming\Python\Python39\site-packages\scipy\linalg\__init__.py)

In [2]:
from numpy import triu

In [6]:
import triu

ModuleNotFoundError: No module named 'triu'

In [2]:
import gensim

In [2]:
from gensim.models import Phrases
from numpy import triu  # Verify numpy works

ImportError: cannot import name 'triu' from 'scipy.linalg.special_matrices' (C:\Users\Claudia\AppData\Roaming\Python\Python39\site-packages\scipy\linalg\special_matrices.py)

In [5]:
from gensim.models import Phrases

ImportError: cannot import name 'triu' from 'scipy.linalg.special_matrices' (C:\Users\Claudia\AppData\Roaming\Python\Python39\site-packages\scipy\linalg\special_matrices.py)

In [185]:
from gensim.models.phrases import Phrases, Phraser

ImportError: cannot import name 'triu' from 'scipy.linalg.special_matrices' (C:\Users\Claudia\AppData\Roaming\Python\Python39\site-packages\scipy\linalg\special_matrices.py)

In [184]:
doc_list=list(new_df_100.filtered_tokens)

In [ ]:
#All together by doc
bigram_list = []
bigram_model = Phrases(doc_list, min_count=100, delimiter="_")

# Iterate over each document in the document list
for doc in doc_list:
    # Apply the bigram model to each document
    for token in bigram_model[doc]:
        # Check if the token is a bigram
        if "_" in token:
            bigram_list.append(token)
            print(token)

In [183]:
from gensim.models.phrases import Phrases, Phraser

def generate_ngrams(doc_list, min_count=5, include_bigrams=True, include_trigrams=False):
    """
    Generate unigrams, bigrams, and trigrams from a list of tokenized documents.
    
    Args:
        doc_list (list of list of str): List of tokenized documents.
        min_count (int): Minimum count for phrases to be included in bigram/trigram models.
        include_bigrams (bool): Whether to include bigrams. Default is True.
        include_trigrams (bool): Whether to include trigrams. Default is False.
        
    Returns:
        list of list of str: List of tokenized documents with n-grams included.
    """
    # Create the bigram model
    bigram_model = Phrases(doc_list, min_count=min_count, delimiter="_")
    bigram_phraser = Phraser(bigram_model)  # Optimized bigram model for fast lookup

    # Apply bigrams to the document list
    doc_list_with_bigrams = [bigram_phraser[doc] for doc in doc_list]

    if include_trigrams:
        # Create the trigram model using the bigram-transformed documents
        trigram_model = Phrases(doc_list_with_bigrams, min_count=min_count, delimiter="_")
        trigram_phraser = Phraser(trigram_model)  # Optimized trigram model

        # Apply trigrams to the bigram-transformed document list
        doc_list_with_trigrams = [trigram_phraser[doc] for doc in doc_list_with_bigrams]
        return doc_list_with_trigrams

    return doc_list_with_bigrams


ImportError: cannot import name 'triu' from 'scipy.linalg.special_matrices' (C:\Users\Claudia\AppData\Roaming\Python\Python39\site-packages\scipy\linalg\special_matrices.py)

In [182]:
from scipy.linalg import triu
print(triu)

ImportError: cannot import name 'triu' from 'scipy.linalg' (C:\Users\Claudia\AppData\Roaming\Python\Python39\site-packages\scipy\linalg\__init__.py)

In [ ]:
# Assuming 'new_df_100' has a column 'cleaned_tokens' with tokenized unigrams
doc_list = list(new_df_100['cleaned_tokens'])

# Generate n-grams (bigrams + unigrams)
ngrams_doc_list = generate_ngrams(doc_list, min_count=5, include_bigrams=True, include_trigrams=False)

# Add the new n-grams to the DataFrame
new_df_100['tokens_with_ngrams'] = ngrams_doc_list

# Display the updated DataFrame
print(new_df_100[['cleaned_tokens', 'tokens_with_ngrams']].head())


In [60]:
from sklearn.feature_extraction.text import CountVectorizer

#CountVect


In [62]:
#only split based on " "
#def split_string(text):
    #tokens = text.split(" ")  
    #return tokens

In [76]:
import re

def split_string(text):
    # Remove all punctuation and special characters
    text = re.sub(r'[^\w\s]', '', text)  # Keep only alphanumeric and spaces
    # Tokenize by splitting on whitespace
    return text.split()

In [77]:
new_df["tokens_text"]= new_df['chunk_text'].apply(split_string)


In [ ]:
#stop words
import nltk
from nltk.corpus import stopwords

# Download the stopwords list if not already downloaded
nltk.download('stopwords')

# Define the stop words
stop_words = set(stopwords.words('english'))

# Function to remove stop words from tokens
def remove_stopwords(tokens):
    return [word for word in tokens if word.lower() not in stop_words]

# Assuming 'new_df' is your DataFrame with a 'tokens' column
new_df['cleaned_tokens'] = new_df['tokens_text'].apply(remove_stopwords)

# Display the updated DataFrame
print(new_df.head())


In [145]:
new_df_100

,district,chunk_index,chunk_text,cleaned_tokens
0,Smethport Area SD,1,smethport area sd district level plan 07/01/20...,"[smethport, area, district, level, plan, distr..."
1,Smethport Area SD,2,"lesson plans, and content resources) aligned w...","[lesson, plan, content, resource, aligned, sta..."
2,Smethport Area SD,3,"pssa grade 4 math, pssa grade 5 math, pssa gra...","[grade, math, grade, math, grade, math, grade,..."
3,Smethport Area SD,4,strategies: common assessment within grade/sub...,"[strategy, common, assessment, within, gradesu..."
4,Smethport Area SD,5,student achievement data to support instructio...,"[student, achievement, data, support, instruct..."
...,...,...,...,...
18850,Davidson County,15,school students metric 3: increase advanced-le...,"[school, student, metric, increase, advancedle..."
18851,Davidson County,16,into instructional design metric 1: visual rep...,"[instructional, design, metric, visual, repres..."
18852,Davidson County,17,schools. members of our community attended mee...,"[school, member, community, attended, meeting,..."
18853,Davidson County,18,our students. this draft was reviewed and revi...,"[student, draft, reviewed, revised, follow, in..."


In [85]:
def remove_numeric_tokens(tokens):
    return [token for token in tokens if not token.isdigit()]

new_df['final_tokens'] = new_df['cleaned_tokens'].apply(remove_numeric_tokens)

In [91]:
#remove unusual words
def remove_unusual_words(tokens):
    # Regular expression to identify words with only alphabets
    pattern = re.compile("^[A-Za-z]+$")

    # Filter out words that don't match the pattern
    cleaned_tokens = [word for word in tokens if pattern.match(word)]

    return cleaned_tokens

# Example usage
tokens = ["built4bibb", "victory", "planned", "2022may", "strategic", "oct"]
cleaned_tokens = remove_unusual_words(tokens)
print(cleaned_tokens)

['victory', 'planned', 'strategic', 'oct']


In [92]:
#oldcode
new_df['final_tokens']=new_df['final_tokens'].apply(remove_unusual_words)

In [ ]:
	Unnamed: 0	chunk	stop	diy_gram	tribigram	min_df	lemma	tfidf	topics
568	568	          150	1	     1          	1	     20      	0	0	10

In [ ]:
#If your dataset has 100 documents, min_df=.20 means a token must appear in at least 20 documents to be included.

In [94]:
# Assuming 'clean_tokens_lemm' is your preprocessed text
new_df['cleaned_text'] = new_df["final_tokens"].apply(lambda x: ' '.join(x))

In [95]:
new_df

,district,chunk_index,chunk_text,tokens_text,cleaned_tokens,final_tokens,cleaned_text
0,Smethport Area SD,1,smethport area sd district level plan 07/01/20...,"[smethport, area, sd, district, level, plan, 0...","[smethport, area, sd, district, level, plan, 0...","[smethport, area, sd, district, level, plan, d...",smethport area sd district level plan district...
1,Smethport Area SD,2,keystone advanced rates at or above the pa sta...,"[keystone, advanced, rates, at, or, above, the...","[keystone, advanced, rates, pa, state, average...","[keystone, advanced, rates, pa, state, average...",keystone advanced rates pa state average math ...
2,Smethport Area SD,3,strategies: common assessment within grade/sub...,"[strategies, common, assessment, within, grade...","[strategies, common, assessment, within, grade...","[strategies, common, assessment, within, grade...",strategies common assessment within gradesubje...
3,Smethport Area SD,4,of testimonials and classroom examples of posi...,"[of, testimonials, and, classroom, examples, o...","[testimonials, classroom, examples, positive, ...","[testimonials, classroom, examples, positive, ...",testimonials classroom examples positive effec...
4,Smethport Area SD,5,safe and supportive schools implementation ste...,"[safe, and, supportive, schools, implementatio...","[safe, supportive, schools, implementation, st...","[safe, supportive, schools, implementation, st...",safe supportive schools implementation steps t...
...,...,...,...,...,...,...,...
12653,Davidson County,9,strengthen the core curriculum at all levels s...,"[strengthen, the, core, curriculum, at, all, l...","[strengthen, core, curriculum, levels, student...","[strengthen, core, curriculum, levels, student...",strengthen core curriculum levels students rea...
12654,Davidson County,10,opportunities for advanced level courses and t...,"[opportunities, for, advanced, level, courses,...","[opportunities, advanced, level, courses, tale...","[opportunities, advanced, level, courses, tale...",opportunities advanced level courses talent de...
12655,Davidson County,11,into instructional design metric 1: visual rep...,"[into, instructional, design, metric, 1, visua...","[instructional, design, metric, 1, visual, rep...","[instructional, design, metric, visual, repres...",instructional design metric visual representat...
12656,Davidson County,12,"sources, was compiled. the initial work of the...","[sources, was, compiled, the, initial, work, o...","[sources, compiled, initial, work, plan, began...","[sources, compiled, initial, work, plan, began...",sources compiled initial work plan began super...


In [96]:
vectorizer = CountVectorizer(lowercase=True, ngram_range=(1,2), min_df=.20)
text_matrix = vectorizer.fit_transform(new_df['cleaned_text'])

In [97]:
print(vectorizer.get_feature_names_out())

['community' 'district' 'learning' 'plan' 'provide' 'school' 'schools'
 'staff' 'strategic' 'student' 'students' 'support']


In [98]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import numpy as np

In [99]:
# Step 2: Define and fit the LDA model
NUM_TOPICS = 10
lda_model = LatentDirichletAllocation(
    n_components=NUM_TOPICS,
    max_iter=1000,
    learning_method='batch',  # You can use 'online' if preferred
    random_state=42,
    evaluate_every=-1,  # No perplexity evaluation during fitting
    verbose=0
)

In [100]:
lda_model.fit(text_matrix)

LatentDirichletAllocation(max_iter=1000, random_state=42)

In [101]:
perplexity = lda_model.perplexity(text_matrix)
print(f"Model Perplexity: {perplexity}")

Model Perplexity: 13.035889658376462


In [102]:
# Step 4: Retrieve Topics
def display_topics(model, feature_names, num_words):
    topics = []
    for topic_idx, topic in enumerate(model.components_):
        top_features_ind = topic.argsort()[:-num_words - 1:-1]
        top_features = [feature_names[i] for i in top_features_ind]
        weights = topic[top_features_ind]
        topics.append(pd.DataFrame({'Word': top_features, 'Weight': weights}))
    return topics

In [103]:
# Get feature names from the CountVectorizer
feature_names = vectorizer.get_feature_names_out()

# Display topics
topics = display_topics(lda_model, feature_names, num_words=15)

In [104]:
# Combine all topics into a single DataFrame
bigdf = pd.concat(topics, axis=1)

# Display the first few columns
print(bigdf.iloc[:, :20])

         Word       Weight       Word       Weight       Word        Weight  \
0     support  4112.631079  strategic  4397.099934   students  12737.099839   
1     provide  3797.099879       plan  3090.669407     school   1164.295422   
2      school     0.100021   district   510.480306    support      0.100017   
3    district     0.100021     school     0.100023   district      0.100017   
4    students     0.100019  community     0.100018    provide      0.100015   
5     student     0.100014   students     0.100017    student      0.100015   
6    learning     0.100012    schools     0.100013  community      0.100014   
7   community     0.100012    student     0.100012   learning      0.100013   
8       staff     0.100012    support     0.100012      staff      0.100010   
9        plan     0.100010    provide     0.100011       plan      0.100010   
10    schools     0.100009      staff     0.100010    schools      0.100009   
11  strategic     0.100006   learning     0.100010  

In [106]:
bigdf.iloc[:, :20]

,Word,Weight,Word,Weight,Word,Weight,Word,Weight,Word,Weight,Word,Weight,Word,Weight,Word,Weight,Word,Weight,Word,Weight
0,support,4112.631079,strategic,4397.099934,students,12737.099839,school,11514.904412,staff,5382.099908,student,7256.099881,learning,6621.099900,schools,4651.099918,community,5219.099877,plan,3714.530504
1,provide,3797.099879,plan,3090.669407,school,1164.295422,district,5019.303021,district,0.100023,school,0.100021,support,396.568793,school,0.100023,district,4626.516535,district,0.100022
2,school,0.100021,district,510.480306,support,0.100017,students,0.100017,school,0.100019,district,0.100019,school,0.100021,district,0.100018,school,0.100020,school,0.100018
3,district,0.100021,school,0.100023,district,0.100017,support,0.100015,students,0.100018,students,0.100019,students,0.100021,students,0.100018,students,0.100019,students,0.100014
4,students,0.100019,community,0.100018,provide,0.100015,plan,0.100014,support,0.100018,support,0.100018,district,0.100018,support,0.100015,support,0.100018,support,0.100014
5,student,0.100014,students,0.100017,student,0.100015,provide,0.100014,provide,0.100016,learning,0.100014,student,0.100017,community,0.100015,provide,0.100013,provide,0.100012
6,learning,0.100012,schools,0.100013,community,0.100014,student,0.100014,community,0.100014,provide,0.100013,provide,0.100016,student,0.100012,staff,0.100013,community,0.100011
7,community,0.100012,student,0.100012,learning,0.100013,community,0.100011,student,0.100013,community,0.100013,community,0.100014,plan,0.100012,plan,0.100013,student,0.100010
8,staff,0.100012,support,0.100012,staff,0.100010,learning,0.100011,plan,0.100011,plan,0.100009,plan,0.100010,provide,0.100011,student,0.100012,staff,0.100009
9,plan,0.100010,provide,0.100011,plan,0.100010,staff,0.100010,learning,0.100010,staff,0.100009,staff,0.100010,learning,0.100010,learning,0.100011,learning,0.100009


In [70]:
print(X.toarray())

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
